# Unit Tests — `services/functions` & `services/ai_agent`

This notebook tests the core calculation and agent logic.  
Run all cells top-to-bottom; the last cell writes `test_results.xlsx`.

In [1]:
import sys, os, math, warnings
import numpy as np
import pandas as pd

# Make sure the project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

warnings.filterwarnings("ignore")
print("Python:", sys.version)
print("Project root:", project_root)

Python: 3.12.12 (main, Jan 27 2026, 23:41:44) [Clang 21.1.4 ]
Project root: /Users/tanut/Desktop/work/Advisor-api--DebtMind-Hackathon


In [2]:
from app.services.functions.AmortiseCal import (
    findInstallment, AmortisationTerm, findRemainOS, findInterestPaid, findTerm,
)
from app.services.functions.util import SummarisePayment, DSR_feasibility, DSR_feasibile_payment
from app.services.ai_agent.GDR_path import generate_GDR_offer
from app.services.ai_agent.nonTDR_extension_path import generate_nonTDR_extension_offer
from app.services.ai_agent.CSL_path import generate_CSL_offer
from app.services.ai_agent.TDR_PL_path import generate_TDR_offer
from app.services.ai_agent.main_agent import DebtSolutionObject
from app.services.model import AdditionalPref
print("All imports OK")

All imports OK


In [3]:
# ── Helpers ────────────────────────────────────────────────────────────────
RESULTS = []

# Numeric columns that must be float so agent code can assign float values back
_FLOAT_COLS = ["os", "installment", "intRate", "accruedInt", "expIntTotal", "actualTerm"]

def make_acc_df(rows: list) -> pd.DataFrame:
    """Create an account DataFrame with float dtypes for columns the agent writes to."""
    df = pd.DataFrame(rows)
    for col in _FLOAT_COLS:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def run_test(group, name, fn, *args, expected=None, tol=0.01, **kwargs):
    try:
        actual = fn(*args, **kwargs)
        if expected is None:
            passed, note = True, "smoke test"
        elif isinstance(expected, bool):
            passed = (actual == expected)
            note = f"expected {expected}, got {actual}"
        elif isinstance(expected, float):
            passed = abs(actual - expected) <= tol
            note = f"expected ≈ {expected}, got {actual}"
        else:
            passed = (actual == expected)
            note = f"expected {expected}, got {actual}"
        status = "PASS" if passed else "FAIL"
    except Exception as e:
        actual, status, note = None, "ERROR", str(e)

    arg_str = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()])
    RESULTS.append({"Group": group, "Test Name": name, "Input": arg_str[:120],
                    "Expected": str(expected) if expected is not None else "-",
                    "Actual": str(actual)[:80] if actual is not None else "-",
                    "Status": status, "Notes": note[:120]})
    print(f"{'✅' if status=='PASS' else ('❌' if status=='FAIL' else '⚠️')} [{group}] {name}: {status}  ({note})")


def run_test_assert(group, name, condition, note=""):
    status = "PASS" if condition else "FAIL"
    RESULTS.append({"Group": group, "Test Name": name, "Input": "-", "Expected": "True",
                    "Actual": str(condition), "Status": status, "Notes": note[:120]})
    print(f"{'✅' if condition else '❌'} [{group}] {name}: {status}  ({note})")

print("Helpers ready")

Helpers ready


---
## 1. `AmortiseCal.findInstallment`

In [4]:
GROUP = "findInstallment"

# 100 000 at 12% p.a. for 12m → PMT ≈ 8884.88 → ceil to nearest 100 = 8900
run_test(GROUP, "standard 12% 12m",  findInstallment, 100_000, 0.12, 12,  expected=8900)
# 200 000 at 7% p.a. for 120m → PMT ≈ 2322.09 → 2400
run_test(GROUP, "low-rate 7% 120m",  findInstallment, 200_000, 0.07, 120, expected=2400)
# 50 000 at 20% p.a. for 60m → PMT ≈ 1323.7 → 1400
run_test(GROUP, "high-rate 20% 60m", findInstallment, 50_000,  0.20, 60,  expected=1400)
# Result is always a multiple of 100
for out_bal, rate, t in [(120_000, 0.09, 48), (80_000, 0.15, 36), (300_000, 0.07, 120)]:
    res = findInstallment(out_bal, rate, t)
    run_test_assert(GROUP, f"multiple-of-100 ({out_bal},{rate},{t})", res % 100 == 0, f"got {res}")

✅ [findInstallment] standard 12% 12m: PASS  (expected 8900, got 8900)
✅ [findInstallment] low-rate 7% 120m: PASS  (expected 2400, got 2400)
✅ [findInstallment] high-rate 20% 60m: PASS  (expected 1400, got 1400)
✅ [findInstallment] multiple-of-100 (120000,0.09,48): PASS  (got 3000)
✅ [findInstallment] multiple-of-100 (80000,0.15,36): PASS  (got 2800)
✅ [findInstallment] multiple-of-100 (300000,0.07,120): PASS  (got 3500)


---
## 2. `AmortiseCal.AmortisationTerm`

In [5]:
GROUP = "AmortisationTerm"

# 100 000 at 1%/m paying 2 000/m → 70 months
run_test(GROUP, "flat-payment convergence",
         AmortisationTerm, 100_000, 0.01, 2_000, expected=70)
# Linear branch: period_payment_increase == period_int
run_test(GROUP, "linear-branch (increase==rate)",
         AmortisationTerm, 100_000, 0.01, 2_000, period_payment_increase=0.01)
# Result must be a positive integer
res = AmortisationTerm(50_000, 0.005, 1_000)
run_test_assert(GROUP, "positive-int result", isinstance(res, int) and res > 0, f"got {res}")

✅ [AmortisationTerm] flat-payment convergence: PASS  (expected 70, got 70)
✅ [AmortisationTerm] linear-branch (increase==rate): PASS  (smoke test)
✅ [AmortisationTerm] positive-int result: PASS  (got 58)


---
## 3. `AmortiseCal.findRemainOS`

In [6]:
GROUP = "findRemainOS"

# After 0 periods → unchanged
res0 = findRemainOS(100_000, 0.01, 2_000, 0)
run_test_assert(GROUP, "0-periods = original OS", abs(res0 - 100_000) < 1, f"got {res0}")
# After full amortisation → near zero (may be slightly negative due to ceil rounding)
term_full = findTerm(100_000, 0.12, 2_000)
res_full  = findRemainOS(100_000, 0.12/12, 2_000, term_full)
run_test_assert(GROUP, "full-term OS near zero", abs(res_full) < 1_000, f"got {res_full:.2f}")
# OS decreases with more payments
r1 = findRemainOS(100_000, 0.01, 2_000, 10)
r2 = findRemainOS(100_000, 0.01, 2_000, 20)
run_test_assert(GROUP, "OS decreases with more payments", r1 > r2, f"{r1:.0f} vs {r2:.0f}")
# Linear branch (increase == rate) → non-negative
res_lin = findRemainOS(50_000, 0.005, 500, 10, period_payment_increase=0.005)
run_test_assert(GROUP, "linear-branch non-negative", res_lin >= 0, f"got {res_lin:.2f}")

✅ [findRemainOS] 0-periods = original OS: PASS  (got 100000.0)
✅ [findRemainOS] full-term OS near zero: PASS  (got -676.34)
✅ [findRemainOS] OS decreases with more payments: PASS  (89538 vs 77981)
✅ [findRemainOS] linear-branch non-negative: PASS  (got 47327.45)


---
## 4. `AmortiseCal.findInterestPaid`

In [7]:
GROUP = "findInterestPaid"

res = findInterestPaid(100_000, 0.01, 2_000, 70)
run_test_assert(GROUP, "total-interest > 0", res > 0, f"got {res:.2f}")
# Higher rate → more interest
int_low  = findInterestPaid(100_000, 0.005, 2_000, 60)
int_high = findInterestPaid(100_000, 0.015, 2_000, 60)
run_test_assert(GROUP, "higher-rate = more-interest", int_high > int_low,
                f"{int_high:.0f} vs {int_low:.0f}")
# Higher payment (shorter term) → less total interest
int_short = findInterestPaid(100_000, 0.01, 3_000, 40)
int_long  = findInterestPaid(100_000, 0.01, 2_000, 70)
run_test_assert(GROUP, "shorter-term = less-interest", int_short < int_long,
                f"{int_short:.0f} vs {int_long:.0f}")
# yearly-increase branch → positive
res_inc = findInterestPaid(100_000, 0.01, 2_000, 24, period_payment_increase=0.05)
run_test_assert(GROUP, "yearly-increase positive", res_inc > 0, f"got {res_inc:.2f}")

✅ [findInterestPaid] total-interest > 0: PASS  (got 39323.66)
✅ [findInterestPaid] higher-rate = more-interest: PASS  (71893 vs 15345)
✅ [findInterestPaid] shorter-term = less-interest: PASS  (22227 vs 39324)
✅ [findInterestPaid] yearly-increase positive: PASS  (got 20958.28)


---
## 5. `AmortiseCal.findTerm`

In [8]:
GROUP = "findTerm"

# Zero OS → 0
run_test(GROUP, "zero-OS returns 0", findTerm, 0, 0.12, 2_000, expected=0)
# Round-trip: findInstallment → findTerm should be ≤ original term
inst60 = findInstallment(100_000, 0.12, 60)
term60 = findTerm(100_000, 0.12, inst60)
run_test_assert(GROUP, "round-trip ≤ 60 months", term60 <= 60, f"term={term60}")
# Higher payment → shorter term
t1 = findTerm(100_000, 0.12, 2_000)
t2 = findTerm(100_000, 0.12, 3_000)
run_test_assert(GROUP, "higher-payment = shorter-term", t1 > t2, f"{t1} vs {t2}")
# yearly_increase → positive
t_inc  = findTerm(100_000, 0.12, 2_000, yearly_increase=0.05)
run_test_assert(GROUP, "yearly-increase positive", t_inc > 0, f"got {t_inc}")
# yearly_increase → shorter than flat
t_flat = findTerm(100_000, 0.12, 2_000)
run_test_assert(GROUP, "yearly-increase shorter than flat", t_inc < t_flat,
                f"{t_inc} vs {t_flat}")

✅ [findTerm] zero-OS returns 0: PASS  (expected 0, got 0)
✅ [findTerm] round-trip ≤ 60 months: PASS  (term=58)
✅ [findTerm] higher-payment = shorter-term: PASS  (70 vs 41)
✅ [findTerm] yearly-increase positive: PASS  (got 62)
✅ [findTerm] yearly-increase shorter than flat: PASS  (62 vs 70)


---
## 6. `util.DSR_feasibility`

In [9]:
GROUP   = "DSR_feasibility"
PUB_OCC = "ข้าราชการ/เจ้าหน้าที่รัฐ (Public Sector)"
PRI_OCC = "พนักงานประจำ (Salaried)"

# Public threshold 0.9 → (30+20+30)/100 = 80% → feasible
run_test(GROUP, "public-sector feasible (80%)",
         DSR_feasibility, 30_000, 20_000, 30_000, 100_000, PUB_OCC, expected=True)
# Public → 92% → not feasible
run_test(GROUP, "public-sector not feasible (92%)",
         DSR_feasibility, 30_000, 30_000, 32_000, 100_000, PUB_OCC, expected=False)
# Private threshold 0.8 → 79% → feasible
run_test(GROUP, "private-sector feasible (79%)",
         DSR_feasibility, 30_000, 20_000, 29_000, 100_000, PRI_OCC, expected=True)
# Private → 85% → not feasible
run_test(GROUP, "private-sector not feasible (85%)",
         DSR_feasibility, 30_000, 30_000, 25_000, 100_000, PRI_OCC, expected=False)
# Exactly at threshold (public, 90%) → not feasible (< not <=)
run_test(GROUP, "exactly at threshold (90%)",
         DSR_feasibility, 30_000, 30_000, 30_000, 100_000, PUB_OCC, expected=False)

✅ [DSR_feasibility] public-sector feasible (80%): PASS  (expected True, got True)
✅ [DSR_feasibility] public-sector not feasible (92%): PASS  (expected False, got False)
✅ [DSR_feasibility] private-sector feasible (79%): PASS  (expected True, got True)
✅ [DSR_feasibility] private-sector not feasible (85%): PASS  (expected False, got False)
✅ [DSR_feasibility] exactly at threshold (90%): PASS  (expected False, got False)


---
## 7. `util.DSR_feasibile_payment`

In [10]:
GROUP   = "DSR_feasibile_payment"
PUB_OCC = "ข้าราชการ/เจ้าหน้าที่รัฐ (Public Sector)"
PRI_OCC = "พนักงานประจำ (Salaried)"

# Public: 0.9*100 000 - 30 000 - 20 000 = 40 000
run_test(GROUP, "public-sector max-payment",
         DSR_feasibile_payment, 30_000, 20_000, 100_000, PUB_OCC,
         expected=40_000.0, tol=0.01)
# Private: 0.8*100 000 - 30 000 - 20 000 = 30 000
run_test(GROUP, "private-sector max-payment",
         DSR_feasibile_payment, 30_000, 20_000, 100_000, PRI_OCC,
         expected=30_000.0, tol=0.01)
# Non-negative when obligations are within threshold
res = DSR_feasibile_payment(10_000, 5_000, 50_000, PRI_OCC)
run_test_assert(GROUP, "non-negative when within threshold", res >= 0, f"got {res}")

✅ [DSR_feasibile_payment] public-sector max-payment: PASS  (expected ≈ 40000.0, got 40000.0)
✅ [DSR_feasibile_payment] private-sector max-payment: PASS  (expected ≈ 30000.0, got 30000.0)
✅ [DSR_feasibile_payment] non-negative when within threshold: PASS  (got 25000.0)


---
## 8. `util.SummarisePayment`

In [11]:
GROUP = "SummarisePayment"

df1 = make_acc_df([{"accNo": "ACC001", "os": 100_000, "intRate": 0.12,
                    "installment": 2_500, "currentDPD": 0, "accruedInt": 1_000, "remainTerm": 60}])
r1 = SummarisePayment(df1)
run_test_assert(GROUP, "returns dict",          isinstance(r1, dict),          type(r1).__name__)
run_test_assert(GROUP, "has TotalOS key",       "TotalOS" in r1,               str(list(r1.keys())))
run_test_assert(GROUP, "TotalOS == 100 000",    r1["TotalOS"] == 100_000,      f"got {r1['TotalOS']}")
run_test_assert(GROUP, "expIntTotal > 0",       r1["expIntTotal"] > 0,         f"got {r1['expIntTotal']:.2f}")
run_test_assert(GROUP, "accNo matches",         r1["accNo"] == "ACC001",       f"got {r1['accNo']}")
run_test_assert(GROUP, "MaxDPD == 0",           r1["MaxDPD"] == 0,             f"got {r1['MaxDPD']}")

# Multi-account
df2 = make_acc_df([
    {"accNo": "A", "os": 50_000, "intRate": 0.12, "installment": 1_500,
     "currentDPD": 5, "accruedInt": 500, "remainTerm": 48},
    {"accNo": "B", "os": 80_000, "intRate": 0.09, "installment": 2_000,
     "currentDPD": 0, "accruedInt": 800, "remainTerm": 60},
])
r2 = SummarisePayment(df2)
run_test_assert(GROUP, "multi-acc TotalOS == 130 000", r2["TotalOS"] == 130_000, f"got {r2['TotalOS']}")
run_test_assert(GROUP, "multi-acc MaxDPD == 5",        r2["MaxDPD"] == 5,        f"got {r2['MaxDPD']}")
run_test_assert(GROUP, "multi-acc accNo sorted",       r2["accNo"] == "A,B",     f"got {r2['accNo']}")

✅ [SummarisePayment] returns dict: PASS  (dict)
✅ [SummarisePayment] has TotalOS key: PASS  (['current_debt', 'accNo', 'expIntTotal', 'MaxDPD', 'TotalOS', 'CurrentInstallment', 'accruedInt', 'MaxIntRate', 'remainTerm', 'actualTerm'])
✅ [SummarisePayment] TotalOS == 100 000: PASS  (got 100000.0)
✅ [SummarisePayment] expIntTotal > 0: PASS  (got 28346.66)
✅ [SummarisePayment] accNo matches: PASS  (got ACC001)
✅ [SummarisePayment] MaxDPD == 0: PASS  (got 0)
✅ [SummarisePayment] multi-acc TotalOS == 130 000: PASS  (got 130000.0)
✅ [SummarisePayment] multi-acc MaxDPD == 5: PASS  (got 5)
✅ [SummarisePayment] multi-acc accNo sorted: PASS  (got A,B)


---
## 9. `ai_agent.GDR_path.generate_GDR_offer`

In [12]:
GROUP = "generate_GDR_offer"

# Eligible: MOB >= 12 (cntrDate 2020 → >6 years old)
df_elig = make_acc_df([{
    "accNo": "G001", "port": "สินเชื่อส่วนบุคคล",
    "os": 100_000, "intRate": 0.12, "installment": 2_500.0,
    "currentDPD": 0, "accruedInt": 1_000, "remainTerm": 60,
    "cntrDate": "1/1/2020", "actualTerm": 60, "expIntTotal": 50_000,
}])
orig_installment = 2_500.0
st_elig = SummarisePayment(df_elig)
offers = generate_GDR_offer(st_elig)
run_test_assert(GROUP, "eligible → 1 offer", len(offers) == 1, f"{len(offers)}")
o = offers[0]
run_test_assert(GROUP, "plan == GDR",                     o.plan == "GDR",          f"{o.plan}")
# During GDR: interest-only payment < original installment
run_test_assert(GROUP, "GDR installment < original (interest-only)",
                o.installment < orig_installment, f"got {o.installment:.2f}")
# After the 3-month holiday, payment resumes at the original amount
run_test_assert(GROUP, "installment_Y2 == original",
                abs(o.installment_Y2 - orig_installment) < 1, f"got {o.installment_Y2}")

# Ineligible: MOB < 12 (just opened May 2026)
df_new = make_acc_df([{
    "accNo": "G002", "port": "สินเชื่อส่วนบุคคล",
    "os": 50_000, "intRate": 0.12, "installment": 1_500.0,
    "currentDPD": 0, "accruedInt": 500, "remainTerm": 36,
    "cntrDate": "5/1/2026", "actualTerm": 36, "expIntTotal": 10_000,
}])
st_new = SummarisePayment(df_new)
run_test_assert(GROUP, "ineligible (MOB<12) → 0 offers",
                len(generate_GDR_offer(st_new)) == 0, "")

✅ [generate_GDR_offer] eligible → 1 offer: PASS  (1)
✅ [generate_GDR_offer] plan == GDR: PASS  (GDR)
✅ [generate_GDR_offer] GDR installment < original (interest-only): PASS  (got 1000.00)
✅ [generate_GDR_offer] installment_Y2 == original: PASS  (got 2500.0)
✅ [generate_GDR_offer] ineligible (MOB<12) → 0 offers: PASS  ()


---
## 10. `ai_agent.nonTDR_extension_path.generate_nonTDR_extension_offer`

In [13]:
GROUP = "generate_nonTDR_extension_offer"

# Eligible: SummarisePayment recalculates actualTerm from findTerm.
# os=80000, rate=12%, installment=3000 → actualTerm ≈ 32.
# remainTerm=60 > 32  → fg_eligible = True → 1 offer generated.
df_ext = make_acc_df([{
    "accNo": "E001", "port": "สินเชื่ออเนกประสงค์",
    "os": 80_000, "intRate": 0.12, "installment": 3_000.0,
    "currentDPD": 0, "accruedInt": 800, "remainTerm": 60,
    "actualTerm": 35.0, "expIntTotal": 20_000,
}])
st_ext = SummarisePayment(df_ext)
offers_ext = generate_nonTDR_extension_offer(st_ext)
run_test_assert(GROUP, "eligible → 1 offer",        len(offers_ext) == 1,           f"{len(offers_ext)}")
run_test_assert(GROUP, "plan == EXT",                offers_ext[0].plan == "EXT",    f"{offers_ext[0].plan}")
run_test_assert(GROUP, "new installment ≤ original",
                offers_ext[0].installment <= 3_000, f"got {offers_ext[0].installment:.2f}")

# Ineligible: remainTerm (25) < recalculated actualTerm (~32) → fg_eligible = False → 0 offers.
# SummarisePayment recalculates actualTerm ≈ 32; with remainTerm=25, 32 >= 25 → not eligible.
df_no_ext = make_acc_df([{
    "accNo": "E002", "port": "สินเชื่ออเนกประสงค์",
    "os": 80_000, "intRate": 0.12, "installment": 3_000.0,
    "currentDPD": 0, "accruedInt": 800, "remainTerm": 25,   # < recalculated actualTerm
    "actualTerm": 65.0, "expIntTotal": 20_000,
}])
st_no_ext = SummarisePayment(df_no_ext)
run_test_assert(GROUP, "ineligible → 0 offers",
                len(generate_nonTDR_extension_offer(st_no_ext)) == 0, "")

✅ [generate_nonTDR_extension_offer] eligible → 1 offer: PASS  (1)
✅ [generate_nonTDR_extension_offer] plan == EXT: PASS  (EXT)
✅ [generate_nonTDR_extension_offer] new installment ≤ original: PASS  (got 1800.00)
✅ [generate_nonTDR_extension_offer] ineligible → 0 offers: PASS  ()


---
## 11. `ai_agent.CSL_path.generate_CSL_offer`

In [14]:
GROUP   = "generate_CSL_offer"
PUB_OCC = "ข้าราชการ/เจ้าหน้าที่รัฐ (Public Sector)"
PRI_OCC = "พนักงานประจำ (Salaried)"

df_csl = make_acc_df([{
    "accNo": "C001", "port": "สินเชื่อส่วนบุคคล",
    "os": 100_000, "intRate": 0.15, "installment": 4_000.0,
    "currentDPD": 0, "accruedInt": 1_000, "remainTerm": 36,
    "actualTerm": 30.0, "expIntTotal": 30_000,
}])
df_KTB = pd.DataFrame([{"accNo": "C001", "installment": 4_000.0, "remainTerm": 36}])
st_csl = SummarisePayment(df_csl)

# Public-sector MOU → 7% interest
user_mou = {"employment_type": PUB_OCC, "EligibleProgram": "สินเชื่ออเนกประสงค์ MOU",
            "age": 40, "IncomeFromSystem": 60_000, "InstallmentNCB_Y1": 5_000}
offers_mou = generate_CSL_offer(st_csl, user_mou, df_KTB, maxPayment=6_000, maxTerm=48)
run_test_assert(GROUP, "MOU public-sector → ≥1 offer", len(offers_mou) >= 1, f"{len(offers_mou)}")
run_test_assert(GROUP, "MOU plan prefix CSL",
                all(o.plan.startswith("CSL") for o in offers_mou),
                str([o.plan for o in offers_mou]))

# No MOU program → SMART MONEY (20%)
user_smart = {"employment_type": PRI_OCC, "EligibleProgram": "",
              "age": 40, "IncomeFromSystem": 100_000, "InstallmentNCB_Y1": 5_000}
offers_smart = generate_CSL_offer(st_csl, user_smart, df_KTB, maxPayment=6_000, maxTerm=48)
run_test_assert(GROUP, "SMART MONEY → ≥1 offer", len(offers_smart) >= 1, f"{len(offers_smart)}")
if offers_smart:
    rate_smart = offers_smart[0].solnAcc[0]["intRate"]
    run_test_assert(GROUP, "SMART MONEY intRate == 0.20",
                    abs(rate_smart - 0.20) < 0.001, f"got {rate_smart}")

ข้าราชการ/เจ้าหน้าที่รัฐ (Public Sector) ['สินเชื่ออเนกประสงค์ MOU']
✅ [generate_CSL_offer] MOU public-sector → ≥1 offer: PASS  (3)
✅ [generate_CSL_offer] MOU plan prefix CSL: PASS  (['CSL01', 'CSL02', 'CSL03'])
พนักงานประจำ (Salaried) ['']
✅ [generate_CSL_offer] SMART MONEY → ≥1 offer: PASS  (3)
✅ [generate_CSL_offer] SMART MONEY intRate == 0.20: PASS  (got 0.2)


---
## 12. `ai_agent.TDR_PL_path.generate_TDR_offer`

In [15]:
GROUP = "generate_TDR_offer"

df_tdr = make_acc_df([{
    "accNo": "T001", "port": "สินเชื่อส่วนบุคคล",   # in PossibleTDRLoan
    "os": 150_000, "intRate": 0.15, "installment": 5_000.0,
    "currentDPD": 5, "accruedInt": 2_000, "remainTerm": 48,
    "actualTerm": 45.0, "expIntTotal": 60_000,
}])
df_ktb = pd.DataFrame([{"accNo": "T001", "installment": 5_000.0, "remainTerm": 48}])
st_tdr = SummarisePayment(df_tdr)
pref   = AdditionalPref(DebtSituation="DebtBurden", refPlanID="",
                         maxPaymentY2=float("inf"), maxPaymentY3=float("inf"))
user_tdr = {"InstallmentNCB_Y1": 3_000.0, "InstallmentNCB_Y2": 3_000.0,
            "InstallmentNCB_Y3": 3_000.0, "CustomerSegment": "C2"}

offers_tdr = generate_TDR_offer(currentStatus=st_tdr, userInfo=user_tdr,
                                 preference=pref.model_dump(),
                                 dfKTBAcc=df_ktb, maxPayment=6_000.0, maxTerm=120)
run_test_assert(GROUP, "eligible PL → ≥1 TDR offer", len(offers_tdr) >= 1, f"{len(offers_tdr)}")
run_test_assert(GROUP, "all plans prefix TDR",
                all(o.plan.startswith("TDR") for o in offers_tdr),
                str([o.plan for o in offers_tdr]))
# TDR01 (min-payment) must always be generated
plan_names = [o.plan for o in offers_tdr]
run_test_assert(GROUP, "TDR01 always included", "TDR01" in plan_names, str(plan_names))

# Non-TDR loan type → 0 offers
df_noTDR = make_acc_df([{
    "accNo": "T002", "port": "สินเชื่อบ้าน",   # not in PossibleTDRLoan
    "os": 150_000, "intRate": 0.15, "installment": 5_000.0,
    "currentDPD": 5, "accruedInt": 2_000, "remainTerm": 48,
    "actualTerm": 45.0, "expIntTotal": 60_000,
}])
st_noTDR = SummarisePayment(df_noTDR)
offers_noTDR = generate_TDR_offer(currentStatus=st_noTDR, userInfo=user_tdr,
                                   preference=pref.model_dump(),
                                   dfKTBAcc=df_ktb, maxPayment=6_000.0, maxTerm=120)
run_test_assert(GROUP, "non-TDR port → 0 offers", len(offers_noTDR) == 0, f"{len(offers_noTDR)}")

✅ [generate_TDR_offer] eligible PL → ≥1 TDR offer: PASS  (6)
✅ [generate_TDR_offer] all plans prefix TDR: PASS  (['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08'])
✅ [generate_TDR_offer] TDR01 always included: PASS  (['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08'])
✅ [generate_TDR_offer] non-TDR port → 0 offers: PASS  (0)


---
## 13. `ai_agent.main_agent.DebtSolutionObject`

In [16]:
GROUP = "DebtSolutionObject"

BASE_ACC = {"accNo": "M001", "port": "สินเชื่อส่วนบุคคล",
            "os": 120_000.0, "intRate": 0.15, "installment": 4_000.0,
            "currentDPD": 0, "accruedInt": 1_200.0, "remainTerm": 48,
            "cntrDate": "1/1/2021"}
BASE_USER = {"CustomerSegment": "C2", "employment_type": "พนักงานประจำ (Salaried)",
             "EligibleProgram": "", "age": 40, "IncomeFromSystem": 80_000,
             "InstallmentNCB_Y1": 3_000, "InstallmentNCB_Y2": 3_000, "InstallmentNCB_Y3": 3_000}

def build_agent(segment="C2", situation="DebtBurden", maxPayment=5_000.0, maxTerm=120,
                df_offerSoln=None, userInfo=None):
    ui = {**BASE_USER, "CustomerSegment": segment}
    if userInfo:
        ui.update(userInfo)
    # Production path: preference arrives as dict via Input.model_dump()
    return DebtSolutionObject(
        userMessage="ต้องการปรับโครงสร้างหนี้", narrative="Customer wants restructure",
        preference=AdditionalPref(DebtSituation=situation, refPlanID="").model_dump(),
        maxPayment=maxPayment, maxTerm=maxTerm,
        userInfo=ui, eligiblePath=[],
        df_offerSoln=df_offerSoln or [],
        dfAccConsult=[BASE_ACC], dfKTBAcc=[BASE_ACC],
    )

# C2 segment → result has expected keys
r = build_agent("C2").get_solution()
run_test_assert(GROUP, "result has replyMessage", "replyMessage" in r,   str(list(r.keys())))
run_test_assert(GROUP, "result has type",         "type" in r,           str(list(r.keys())))
run_test_assert(GROUP, "result has newOfferSoln", "newOfferSoln" in r,   str(list(r.keys())))
run_test_assert(GROUP, "shortlisted ≤ 2 offers",  len(r["newOfferSoln"]) <= 2,
                f"got {len(r['newOfferSoln'])}")

# Current segment: can produce CSL + TDR mix
r_curr = build_agent("Current").get_solution()
run_test_assert(GROUP, "Current segment returns solution",
                "replyMessage" in r_curr, str(list(r_curr.keys())))

# Repeat run: check_repeat filters exact duplicates by (plan, refAccNo, term, installment).
# Different plans from the same family still surface → assert original plans are NOT re-offered.
first      = build_agent("C2").get_solution()
first_plans = {o["plan"] for o in first["newOfferSoln"]}
r2         = build_agent("C2", df_offerSoln=first["newOfferSoln"]).get_solution()
r2_plans   = {o["plan"] for o in r2["newOfferSoln"]}
run_test_assert(GROUP, "repeat run → no re-offer of same plans",
                len(first_plans & r2_plans) == 0,
                f"overlap: {first_plans & r2_plans}")

# C1-X segment → can use CSL
r_c1x = build_agent("C1-X").get_solution()
run_test_assert(GROUP, "C1-X segment returns solution",
                "replyMessage" in r_c1x, str(list(r_c1x.keys())))

['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] result has replyMessage: PASS  (['newOfferSoln', 'newOfferSolnAcc', 'newOfferCard', 'replyMessage', 'type'])
✅ [DebtSolutionObject] result has type: PASS  (['newOfferSoln', 'newOfferSolnAcc', 'newOfferCard', 'replyMessage', 'type'])
✅ [DebtSolutionObject] result has newOfferSoln: PASS  (['newOfferSoln', 'newOfferSolnAcc', 'newOfferCard', 'replyMessage', 'type'])
✅ [DebtSolutionObject] shortlisted ≤ 2 offers: PASS  (got 2)
พนักงานประจำ (Salaried) ['']
['CSL01', 'CSL02', 'EXT', 'GDR', 'TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] Current segment returns solution: PASS  (['newOfferSoln', 'newOfferSolnAcc', 'newOfferCard', 'replyMessage', 'type'])
['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
['TDR01', 'TDR03', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] repeat run → no re-offer of same plans: PASS  (overlap: set())
พนักงานประจำ (Salaried) ['']
['CSL01', 'CSL02', 'EXT', 'GDR',

---
## 1b. `findInstallment` — extended

In [17]:
GROUP = "findInstallment"

# Known values
run_test(GROUP, "5% 36m",              findInstallment, 100_000, 0.05, 36,  expected=3000)
run_test(GROUP, "18% 24m",             findInstallment, 50_000,  0.18, 24,  expected=2500)
run_test(GROUP, "single-month 12% 1m", findInstallment, 10_000,  0.12, 1,   expected=10100)
run_test(GROUP, "9% 60m",              findInstallment, 75_000,  0.09, 60,  expected=1600)
run_test(GROUP, "6% 240m",             findInstallment, 200_000, 0.06, 240, expected=1500)

# Property: higher rate → higher installment (same OS, same term)
i_low  = findInstallment(100_000, 0.12, 12)
i_high = findInstallment(100_000, 0.18, 12)
run_test_assert(GROUP, "higher-rate = higher-installment", i_high > i_low, f"{i_high} vs {i_low}")

# Property: longer term → lower installment (same OS, same rate)
i_short = findInstallment(100_000, 0.12, 12)
i_long  = findInstallment(100_000, 0.12, 120)
run_test_assert(GROUP, "longer-term = lower-installment", i_long < i_short, f"{i_long} vs {i_short}")

# Property: installment × term ≥ OS (total payments cover principal)
for _os, _rate, _term in [(80_000, 0.10, 48), (500_000, 0.07, 180), (15_000, 0.20, 12)]:
    inst = findInstallment(_os, _rate, _term)
    run_test_assert(GROUP, f"total-payments >= OS ({_os},{_rate},{_term})",
                    inst * _term >= _os, f"{inst}×{_term}={inst*_term} vs OS={_os}")

✅ [findInstallment] 5% 36m: PASS  (expected 3000, got 3000)
✅ [findInstallment] 18% 24m: PASS  (expected 2500, got 2500)
✅ [findInstallment] single-month 12% 1m: PASS  (expected 10100, got 10100)
✅ [findInstallment] 9% 60m: PASS  (expected 1600, got 1600)
✅ [findInstallment] 6% 240m: PASS  (expected 1500, got 1500)
✅ [findInstallment] higher-rate = higher-installment: PASS  (9200 vs 8900)
✅ [findInstallment] longer-term = lower-installment: PASS  (1500 vs 8900)
✅ [findInstallment] total-payments >= OS (80000,0.1,48): PASS  (2100×48=100800 vs OS=80000)
✅ [findInstallment] total-payments >= OS (500000,0.07,180): PASS  (4500×180=810000 vs OS=500000)
✅ [findInstallment] total-payments >= OS (15000,0.2,12): PASS  (1400×12=16800 vs OS=15000)


---
## 2b. `AmortisationTerm` — extended

In [18]:
GROUP = "AmortisationTerm"

# Small OS: quick payoff
run_test(GROUP, "small-OS quick-payoff",
         AmortisationTerm, 10_000, 0.01, 1_000, expected=11)

# Higher rate (same payment) → longer term
t_low  = AmortisationTerm(100_000, 0.005, 2_000)
t_high = AmortisationTerm(100_000, 0.015, 2_000)
run_test_assert(GROUP, "higher-rate = longer-term", t_high > t_low, f"{t_high} vs {t_low}")

# Larger payment → shorter term (with increase)
t_less = AmortisationTerm(100_000, 0.01, 2_000, 0.005)
t_more = AmortisationTerm(100_000, 0.01, 3_000, 0.005)
run_test_assert(GROUP, "larger-payment shorter-term (with increase)", t_less > t_more, f"{t_less} vs {t_more}")

# period_payment_increase = 0.005: result is positive integer
res_inc = AmortisationTerm(100_000, 0.01, 2_000, 0.005)
run_test_assert(GROUP, "non-zero-increase positive-int", isinstance(res_inc, int) and res_inc > 0, f"{res_inc}")

# Term must be ≥ 1 for any OS > interest
t_min = AmortisationTerm(5_000, 0.01, 4_000)
run_test_assert(GROUP, "term ≥ 1",  t_min >= 1, f"got {t_min}")

✅ [AmortisationTerm] small-OS quick-payoff: PASS  (expected 11, got 11)
✅ [AmortisationTerm] higher-rate = longer-term: PASS  (94 vs 58)
✅ [AmortisationTerm] larger-payment shorter-term (with increase): PASS  (58 vs 37)
✅ [AmortisationTerm] non-zero-increase positive-int: PASS  (58)
✅ [AmortisationTerm] term ≥ 1: PASS  (got 2)


---
## 3b. `findRemainOS` — extended

In [19]:
GROUP = "findRemainOS"

# Zero period_int → linear: OS - payment × periods
r_zero = findRemainOS(100_000, 0, 2_000, 10)
run_test_assert(GROUP, "zero-rate linear branch",
                abs(r_zero - 80_000) < 1, f"got {r_zero}")

# Single-period payment
r1 = findRemainOS(100_000, 0.01, 2_000, 1)
run_test_assert(GROUP, "1-period: os×(1+r) - payment",
                abs(r1 - (100_000 * 1.01 - 2_000)) < 0.01, f"got {r1:.2f}")

# Monotone across three checkpoints
r_10 = findRemainOS(100_000, 0.01, 2_000, 10)
r_30 = findRemainOS(100_000, 0.01, 2_000, 30)
r_50 = findRemainOS(100_000, 0.01, 2_000, 50)
run_test_assert(GROUP, "OS monotone: 10 > 30 > 50 periods",
                r_10 > r_30 > r_50, f"{r_10:.0f} > {r_30:.0f} > {r_50:.0f}")

# Higher payment → lower remaining OS after same periods
r_low  = findRemainOS(100_000, 0.01, 2_000, 20)
r_high = findRemainOS(100_000, 0.01, 3_000, 20)
run_test_assert(GROUP, "higher-payment = lower-OS",
                r_high < r_low, f"{r_high:.0f} vs {r_low:.0f}")

# Non-zero increase branch: result ≥ 0 at early payoff
r_inc = findRemainOS(100_000, 0.01, 2_000, 5, period_payment_increase=0.005)
run_test_assert(GROUP, "increase-branch early: non-negative",
                r_inc >= 0, f"got {r_inc:.2f}")

✅ [findRemainOS] zero-rate linear branch: PASS  (got 80000)
✅ [findRemainOS] 1-period: os×(1+r) - payment: PASS  (got 99000.00)
✅ [findRemainOS] OS monotone: 10 > 30 > 50 periods: PASS  (89538 > 65215 > 35537)
✅ [findRemainOS] higher-payment = lower-OS: PASS  (55962 vs 77981)
✅ [findRemainOS] increase-branch early: non-negative: PASS  (got 94797.49)


---
## 4b. `findInterestPaid` — extended

In [20]:
GROUP = "findInterestPaid"

# Large payment → less interest
int_small_pay = findInterestPaid(100_000, 0.01, 2_000, 70)
int_large_pay = findInterestPaid(100_000, 0.01, 5_000, findTerm(100_000, 0.12, 5_000))
run_test_assert(GROUP, "larger-payment = less-interest",
                int_large_pay < int_small_pay, f"{int_large_pay:.0f} vs {int_small_pay:.0f}")

# More periods → more total interest paid
i_40 = findInterestPaid(100_000, 0.01, 2_000, 40)
i_70 = findInterestPaid(100_000, 0.01, 2_000, 70)
run_test_assert(GROUP, "more-periods = more-interest",
                i_70 > i_40, f"{i_70:.0f} vs {i_40:.0f}")

# Interest always positive for any valid loan
for _os, _rate, _pay, _periods in [(50_000, 0.005, 1_000, 60), (200_000, 0.01, 4_000, 70)]:
    i = findInterestPaid(_os, _rate, _pay, _periods)
    run_test_assert(GROUP, f"positive-interest ({_os},{_rate})", i > 0, f"got {i:.2f}")

# Round-trip: total_payments - OS ≈ interest (within rounding)
term70 = findTerm(100_000, 0.12, 2_000)
int70  = findInterestPaid(100_000, 0.12/12, 2_000, term70)
run_test_assert(GROUP, "round-trip: total_paid - OS ≈ interest",
                abs(term70 * 2_000 - 100_000 - int70) < 1_000,
                f"diff={abs(term70*2_000 - 100_000 - int70):.2f}")

✅ [findInterestPaid] larger-payment = less-interest: PASS  (12135 vs 39324)
✅ [findInterestPaid] more-periods = more-interest: PASS  (39324 vs 31114)
✅ [findInterestPaid] positive-interest (50000,0.005): PASS  (got 7672.48)
✅ [findInterestPaid] positive-interest (200000,0.01): PASS  (got 78647.33)
✅ [findInterestPaid] round-trip: total_paid - OS ≈ interest: PASS  (diff=676.34)


---
## 5b. `findTerm` — extended

In [21]:
GROUP = "findTerm"

# Small OS → short term
run_test(GROUP, "small-OS short-term", findTerm, 5_000, 0.12, 2_000, expected=3)

# Payment >> OS → 1 month
run_test(GROUP, "overpayment → 1 month", findTerm, 10_000, 0.12, 20_000, expected=1)

# Larger OS → longer term (same rate and payment)
t_small = findTerm(50_000,  0.12, 2_000)
t_large = findTerm(150_000, 0.12, 2_000)
run_test_assert(GROUP, "larger-OS = longer-term", t_large > t_small, f"{t_large} vs {t_small}")

# Higher yearly_increase → shorter term
t_yi_lo = findTerm(100_000, 0.12, 2_000, yearly_increase=0.03)
t_yi_hi = findTerm(100_000, 0.12, 2_000, yearly_increase=0.10)
run_test_assert(GROUP, "higher-yearly-increase shorter-term",
                t_yi_hi < t_yi_lo, f"{t_yi_hi} vs {t_yi_lo}")

# Both yearly_increase variants return positive terms
for yi in [0.02, 0.05, 0.08]:
    t = findTerm(100_000, 0.12, 2_000, yearly_increase=yi)
    run_test_assert(GROUP, f"yearly-increase={yi} positive", t > 0, f"got {t}")

✅ [findTerm] small-OS short-term: PASS  (expected 3, got 3)
✅ [findTerm] overpayment → 1 month: PASS  (expected 1, got 1)
✅ [findTerm] larger-OS = longer-term: PASS  (140 vs 29)
✅ [findTerm] higher-yearly-increase shorter-term: PASS  (56 vs 65)
✅ [findTerm] yearly-increase=0.02 positive: PASS  (got 66)
✅ [findTerm] yearly-increase=0.05 positive: PASS  (got 62)
✅ [findTerm] yearly-increase=0.08 positive: PASS  (got 58)


---
## 6b. `DSR_feasibility` — extended

In [22]:
GROUP   = "DSR_feasibility"
PUB_OCC  = "ข้าราชการ/เจ้าหน้าที่รัฐ (Public Sector)"
PRI_OCC  = "พนักงานประจำ (Salaried)"
SELF_OCC = "ประกอบธุรกิจส่วนตัว (Self-employed)"

# Unknown/self-employed occupation → 0.8 threshold
run_test(GROUP, "self-employed (0.8 threshold) feasible (75%)",
         DSR_feasibility, 30_000, 20_000, 25_000, 100_000, SELF_OCC, expected=True)
run_test(GROUP, "self-employed not feasible (82%)",
         DSR_feasibility, 30_000, 30_000, 22_000, 100_000, SELF_OCC, expected=False)

# Exactly at private threshold (80%) → not feasible (< not <=)
run_test(GROUP, "exactly 80% private → not feasible",
         DSR_feasibility, 0, 0, 80_000, 100_000, PRI_OCC, expected=False)
# Just under (79.9%) → feasible
run_test(GROUP, "79.9% private → feasible",
         DSR_feasibility, 0, 0, 79_900, 100_000, PRI_OCC, expected=True)

# Zero KTB installment: public 85% → feasible
run_test(GROUP, "zero-KTB public feasible (85%)",
         DSR_feasibility, 0, 50_000, 35_000, 100_000, PUB_OCC, expected=True)

# Zero NCB installment: private 50% → feasible
run_test(GROUP, "zero-NCB private feasible (50%)",
         DSR_feasibility, 50_000, 0, 0, 100_000, PRI_OCC, expected=True)

✅ [DSR_feasibility] self-employed (0.8 threshold) feasible (75%): PASS  (expected True, got True)
✅ [DSR_feasibility] self-employed not feasible (82%): PASS  (expected False, got False)
✅ [DSR_feasibility] exactly 80% private → not feasible: PASS  (expected False, got False)
✅ [DSR_feasibility] 79.9% private → feasible: PASS  (expected True, got True)
✅ [DSR_feasibility] zero-KTB public feasible (85%): PASS  (expected True, got True)
✅ [DSR_feasibility] zero-NCB private feasible (50%): PASS  (expected True, got True)


---
## 7b. `DSR_feasibile_payment` — extended

In [23]:
GROUP   = "DSR_feasibile_payment"
PUB_OCC  = "ข้าราชการ/เจ้าหน้าที่รัฐ (Public Sector)"
PRI_OCC  = "พนักงานประจำ (Salaried)"
SELF_OCC = "ประกอบธุรกิจส่วนตัว (Self-employed)"

# Zero obligations: public → 0.9 × income
run_test(GROUP, "zero-obligations public → 90% income",
         DSR_feasibile_payment, 0, 0, 100_000, PUB_OCC, expected=90_000.0, tol=0.01)

# Unknown occ → 0.8 threshold
run_test(GROUP, "self-employed (0.8) max-payment",
         DSR_feasibile_payment, 0, 0, 100_000, SELF_OCC, expected=80_000.0, tol=0.01)

# High income: public: 0.9×500_000 - 100_000 - 50_000 = 300_000
run_test(GROUP, "high-income public max-payment",
         DSR_feasibile_payment, 100_000, 50_000, 500_000, PUB_OCC, expected=300_000.0, tol=0.01)

# Result is always a float
res = DSR_feasibile_payment(20_000, 10_000, 100_000, PRI_OCC)
run_test_assert(GROUP, "result is float", isinstance(res, float), type(res).__name__)

✅ [DSR_feasibile_payment] zero-obligations public → 90% income: PASS  (expected ≈ 90000.0, got 90000.0)
✅ [DSR_feasibile_payment] self-employed (0.8) max-payment: PASS  (expected ≈ 80000.0, got 80000.0)
✅ [DSR_feasibile_payment] high-income public max-payment: PASS  (expected ≈ 300000.0, got 300000.0)
✅ [DSR_feasibile_payment] result is float: PASS  (float)


---
## 8b. `SummarisePayment` — extended

In [24]:
GROUP = "SummarisePayment"

# Three-account aggregation
df3 = make_acc_df([
    {"accNo": "A", "os": 30_000, "intRate": 0.12, "installment": 1_000,
     "currentDPD": 0, "accruedInt": 300, "remainTerm": 36},
    {"accNo": "B", "os": 50_000, "intRate": 0.09, "installment": 1_500,
     "currentDPD": 3, "accruedInt": 500, "remainTerm": 48},
    {"accNo": "C", "os": 70_000, "intRate": 0.15, "installment": 2_000,
     "currentDPD": 0, "accruedInt": 700, "remainTerm": 60},
])
r3 = SummarisePayment(df3)
run_test_assert(GROUP, "3-acc TotalOS == 150 000",       r3["TotalOS"]  == 150_000,  f"got {r3['TotalOS']}")
run_test_assert(GROUP, "3-acc MaxDPD == 3",              r3["MaxDPD"]   == 3,        f"got {r3['MaxDPD']}")
run_test_assert(GROUP, "3-acc accNo alphabetical A,B,C", r3["accNo"]    == "A,B,C",  f"got {r3['accNo']}")
run_test_assert(GROUP, "3-acc MaxIntRate == 0.15",       r3["MaxIntRate"] == 0.15,   f"got {r3['MaxIntRate']}")
run_test_assert(GROUP, "3-acc CurrentInstallment == 4500",
                r3["CurrentInstallment"] == 4_500, f"got {r3['CurrentInstallment']}")
run_test_assert(GROUP, "actualTerm key present",         "actualTerm" in r3, str(list(r3.keys())))
run_test_assert(GROUP, "actualTerm > 0",                 r3["actualTerm"] > 0, f"got {r3['actualTerm']}")

✅ [SummarisePayment] 3-acc TotalOS == 150 000: PASS  (got 150000.0)
✅ [SummarisePayment] 3-acc MaxDPD == 3: PASS  (got 3)
✅ [SummarisePayment] 3-acc accNo alphabetical A,B,C: PASS  (got A,B,C)
✅ [SummarisePayment] 3-acc MaxIntRate == 0.15: PASS  (got 0.15)
✅ [SummarisePayment] 3-acc CurrentInstallment == 4500: PASS  (got 4500.0)
✅ [SummarisePayment] actualTerm key present: PASS  (['current_debt', 'accNo', 'expIntTotal', 'MaxDPD', 'TotalOS', 'CurrentInstallment', 'accruedInt', 'MaxIntRate', 'remainTerm', 'actualTerm'])
✅ [SummarisePayment] actualTerm > 0: PASS  (got 47)


---
## 9b. `generate_GDR_offer` — extended

In [25]:
GROUP = "generate_GDR_offer"

# Very old account (2015) → eligible, MOB >> 12
df_old = make_acc_df([{
    "accNo": "G010", "port": "สินเชื่อส่วนบุคคล",
    "os": 60_000, "intRate": 0.15, "installment": 2_000.0,
    "currentDPD": 0, "accruedInt": 600, "remainTerm": 40,
    "cntrDate": "1/1/2015", "actualTerm": 38.0, "expIntTotal": 15_000,
}])
st_old = SummarisePayment(df_old)
offers_old = generate_GDR_offer(st_old)
run_test_assert(GROUP, "very-old MOB→ eligible 1 offer", len(offers_old) == 1, f"{len(offers_old)}")

# GDR offer: interest paid increases (3-month interest holiday adds cost)
orig_int = st_old["expIntTotal"]
gdr_int  = offers_old[0].totalIntPaid
run_test_assert(GROUP, "GDR total-int > original (3m interest added)",
                gdr_int > orig_int, f"gdr={gdr_int:.2f} vs orig={orig_int:.2f}")

# Multi-account: 1 eligible (old) + 1 ineligible (new) → 1 offer with mixed accounts
df_mixed = make_acc_df([
    {"accNo": "G020", "port": "สินเชื่อส่วนบุคคล", "os": 50_000, "intRate": 0.12,
     "installment": 2_000.0, "currentDPD": 0, "accruedInt": 500, "remainTerm": 36,
     "cntrDate": "1/1/2020", "actualTerm": 30.0, "expIntTotal": 10_000},
    {"accNo": "G021", "port": "สินเชื่ออเนกประสงค์", "os": 80_000, "intRate": 0.15,
     "installment": 3_000.0, "currentDPD": 0, "accruedInt": 800, "remainTerm": 48,
     "cntrDate": "5/1/2026", "actualTerm": 45.0, "expIntTotal": 20_000},
])
st_mixed = SummarisePayment(df_mixed)
offers_mixed = generate_GDR_offer(st_mixed)
run_test_assert(GROUP, "mixed-eligible → 1 offer", len(offers_mixed) == 1, f"{len(offers_mixed)}")

# offerCard is a Pydantic OfferCard object (not a plain dict)
o_card = offers_old[0].offerCard
run_test_assert(GROUP, "offer has offerCard (not None)", o_card is not None, str(type(o_card)))
run_test_assert(GROUP, "offerCard has plan_id attr", hasattr(o_card, "plan_id"), str(type(o_card)))

✅ [generate_GDR_offer] very-old MOB→ eligible 1 offer: PASS  (1)
✅ [generate_GDR_offer] GDR total-int > original (3m interest added): PASS  (gdr=17921.32 vs orig=15671.32)
✅ [generate_GDR_offer] mixed-eligible → 1 offer: PASS  (1)
✅ [generate_GDR_offer] offer has offerCard (not None): PASS  (<class 'app.services.HTML.model_HTML.OfferCard'>)
✅ [generate_GDR_offer] offerCard has plan_id attr: PASS  (<class 'app.services.HTML.model_HTML.OfferCard'>)


---
## 10b. `generate_nonTDR_extension_offer` — extended

In [26]:
GROUP = "generate_nonTDR_extension_offer"

# actualTerm == remainTerm (recalculated) → ineligible
df_eq = make_acc_df([{
    "accNo": "E010", "port": "สินเชื่ออเนกประสงค์",
    "os": 80_000, "intRate": 0.12, "installment": 3_000.0,
    "currentDPD": 0, "accruedInt": 800, "remainTerm": 32,
    "actualTerm": 35.0, "expIntTotal": 20_000,   # SummarisePayment recalculates to ≈32
}])
st_eq = SummarisePayment(df_eq)
run_test_assert(GROUP, "actualTerm==remainTerm → 0 offers",
                len(generate_nonTDR_extension_offer(st_eq)) == 0, "")

# High payment → short actualTerm, long remainTerm → eligible + lower new installment
df_hi = make_acc_df([{
    "accNo": "E011", "port": "สินเชื่ออเนกประสงค์",
    "os": 50_000, "intRate": 0.09, "installment": 5_000.0,
    "currentDPD": 0, "accruedInt": 500, "remainTerm": 60,
    "actualTerm": 60.0, "expIntTotal": 10_000,
}])
st_hi = SummarisePayment(df_hi)
offers_hi = generate_nonTDR_extension_offer(st_hi)
run_test_assert(GROUP, "high-payment short-actual → eligible", len(offers_hi) == 1, f"{len(offers_hi)}")
run_test_assert(GROUP, "new installment < original 5000",
                offers_hi[0].installment < 5_000, f"got {offers_hi[0].installment:.2f}")

# Offer solnAcc is a list
run_test_assert(GROUP, "solnAcc is list",
                isinstance(offers_hi[0].solnAcc, list), type(offers_hi[0].solnAcc).__name__)

✅ [generate_nonTDR_extension_offer] actualTerm==remainTerm → 0 offers: PASS  ()
✅ [generate_nonTDR_extension_offer] high-payment short-actual → eligible: PASS  (1)
✅ [generate_nonTDR_extension_offer] new installment < original 5000: PASS  (got 1100.00)
✅ [generate_nonTDR_extension_offer] solnAcc is list: PASS  (list)


---
## 11b. `generate_TDR_offer` — extended

In [27]:
GROUP = "generate_TDR_offer"

df_tdr2 = make_acc_df([{
    "accNo": "T010", "port": "สินเชื่อส่วนบุคคล",
    "os": 100_000, "intRate": 0.12, "installment": 4_000.0,
    "currentDPD": 0, "accruedInt": 1_000, "remainTerm": 48,
    "actualTerm": 35.0, "expIntTotal": 20_000,
}])
df_ktb2 = pd.DataFrame([{"accNo": "T010", "installment": 4_000.0, "remainTerm": 48}])
st2     = SummarisePayment(df_tdr2)
pref2   = AdditionalPref(DebtSituation="DebtBurden", refPlanID="",
                          maxPaymentY2=float("inf"), maxPaymentY3=float("inf"))
user2   = {"InstallmentNCB_Y1": 3_000.0, "InstallmentNCB_Y2": 3_000.0,
           "InstallmentNCB_Y3": 3_000.0, "CustomerSegment": "C2"}

# TDR10 generated when maxTerm < 120
offers_60 = generate_TDR_offer(currentStatus=st2, userInfo=user2,
                                preference=pref2.model_dump(),
                                dfKTBAcc=df_ktb2, maxPayment=5_000.0, maxTerm=60)
plan_names_60 = [o.plan for o in offers_60]
run_test_assert(GROUP, "maxTerm=60 → TDR10 present", "TDR10" in plan_names_60, str(plan_names_60))

# TDR10 not present when maxTerm == 120 (default)
offers_120 = generate_TDR_offer(currentStatus=st2, userInfo=user2,
                                 preference=pref2.model_dump(),
                                 dfKTBAcc=df_ktb2, maxPayment=5_000.0, maxTerm=120)
plan_names_120 = [o.plan for o in offers_120]
run_test_assert(GROUP, "maxTerm=120 → TDR10 absent", "TDR10" not in plan_names_120, str(plan_names_120))

# Non-TDR-eligible port mixed with TDR port → only TDR ports generate offers
df_mixed_tdr = make_acc_df([
    {"accNo": "T020", "port": "สินเชื่อส่วนบุคคล",  # eligible
     "os": 100_000, "intRate": 0.12, "installment": 4_000.0,
     "currentDPD": 0, "accruedInt": 1_000, "remainTerm": 48, "actualTerm": 35.0, "expIntTotal": 20_000},
    {"accNo": "T021", "port": "สินเชื่อบ้าน",        # not eligible
     "os": 500_000, "intRate": 0.05, "installment": 5_000.0,
     "currentDPD": 0, "accruedInt": 2_000, "remainTerm": 120, "actualTerm": 110.0, "expIntTotal": 100_000},
])
df_ktb_mix = pd.DataFrame([
    {"accNo": "T020", "installment": 4_000.0, "remainTerm": 48},
    {"accNo": "T021", "installment": 5_000.0, "remainTerm": 120},
])
st_mix = SummarisePayment(df_mixed_tdr)
offers_mix = generate_TDR_offer(currentStatus=st_mix, userInfo=user2,
                                 preference=pref2.model_dump(),
                                 dfKTBAcc=df_ktb_mix, maxPayment=8_000.0, maxTerm=120)
run_test_assert(GROUP, "mixed-port → ≥1 offer (T020 eligible)", len(offers_mix) >= 1, f"{len(offers_mix)}")
run_test_assert(GROUP, "all offers have solnAcc list",
                all(isinstance(o.solnAcc, list) for o in offers_mix), "")

✅ [generate_TDR_offer] maxTerm=60 → TDR10 present: PASS  (['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08', 'TDR10'])
✅ [generate_TDR_offer] maxTerm=120 → TDR10 absent: PASS  (['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08'])
✅ [generate_TDR_offer] mixed-port → ≥1 offer (T020 eligible): PASS  (5)
✅ [generate_TDR_offer] all offers have solnAcc list: PASS  ()


---
## 12b. `DebtSolutionObject` — extended situations & segments

In [28]:
GROUP = "DebtSolutionObject"

# C3 segment (TDR-focused)
r_c3 = build_agent("C3").get_solution()
run_test_assert(GROUP, "C3 segment → solution",    "replyMessage" in r_c3, str(list(r_c3.keys())))
run_test_assert(GROUP, "C3 result type == json",   r_c3.get("type") == "json", r_c3.get("type"))
run_test_assert(GROUP, "C3 ≤ 2 offers",           len(r_c3["newOfferSoln"]) <= 2, f"{len(r_c3['newOfferSoln'])}")

# DebtSituation: TemporaryCashflow
r_tc = build_agent("C2", "TemporaryCashflow").get_solution()
run_test_assert(GROUP, "TemporaryCashflow → solution", "replyMessage" in r_tc, "")
run_test_assert(GROUP, "TemporaryCashflow ≤ 2 offers", len(r_tc["newOfferSoln"]) <= 2,
                f"{len(r_tc['newOfferSoln'])}")

# DebtSituation: PermanentAffordability
r_pa = build_agent("C2", "PermanentAffordability").get_solution()
run_test_assert(GROUP, "PermanentAffordability → solution", "replyMessage" in r_pa, "")

# DebtSituation: FinancialShock (Current segment gets GDR)
r_fs = build_agent("Current", "FinancialShock").get_solution()
run_test_assert(GROUP, "FinancialShock → solution", "replyMessage" in r_fs, "")

# DebtSituation: CareerChange
r_cc = build_agent("C2", "CareerChange").get_solution()
run_test_assert(GROUP, "CareerChange → solution", "replyMessage" in r_cc, "")

# newOfferSolnAcc and newOfferCard keys present
run_test_assert(GROUP, "newOfferSolnAcc key present", "newOfferSolnAcc" in r_c3, str(list(r_c3.keys())))
run_test_assert(GROUP, "newOfferCard key present",    "newOfferCard"    in r_c3, str(list(r_c3.keys())))

['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] C3 segment → solution: PASS  (['newOfferSoln', 'newOfferSolnAcc', 'newOfferCard', 'replyMessage', 'type'])
✅ [DebtSolutionObject] C3 result type == json: PASS  (json)
✅ [DebtSolutionObject] C3 ≤ 2 offers: PASS  (2)
['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] TemporaryCashflow → solution: PASS  ()
✅ [DebtSolutionObject] TemporaryCashflow ≤ 2 offers: PASS  (2)
['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] PermanentAffordability → solution: PASS  ()
พนักงานประจำ (Salaried) ['']
['CSL01', 'CSL02', 'EXT', 'GDR', 'TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] FinancialShock → solution: PASS  ()
['TDR01', 'TDR02', 'TDR03', 'TDR04', 'TDR06', 'TDR08']
✅ [DebtSolutionObject] CareerChange → solution: PASS  ()
✅ [DebtSolutionObject] newOfferSolnAcc key present: PASS  (['newOfferSoln', 'newOfferSolnAcc', 'newOfferCard', 'replyM

---
## Summary

In [29]:
df_results = pd.DataFrame(RESULTS)
total  = len(df_results)
passed = (df_results["Status"] == "PASS").sum()
failed = (df_results["Status"] == "FAIL").sum()
errors = (df_results["Status"] == "ERROR").sum()

print(f"\n{'='*55}")
print(f"  Total : {total:3d}")
print(f"  PASS  : {passed:3d}  ({100*passed/total:.1f}%)")
print(f"  FAIL  : {failed:3d}")
print(f"  ERROR : {errors:3d}")
print(f"{'='*55}")
print("\nBy group:")
print(df_results.groupby("Group")["Status"]
      .value_counts().unstack(fill_value=0).to_string())


  Total : 135
  PASS  : 135  (100.0%)
  FAIL  :   0
  ERROR :   0

By group:
Status                           PASS
Group                                
AmortisationTerm                    8
DSR_feasibile_payment               7
DSR_feasibility                    11
DebtSolutionObject                 17
SummarisePayment                   16
findInstallment                    16
findInterestPaid                    9
findRemainOS                        9
findTerm                           12
generate_CSL_offer                  4
generate_GDR_offer                 10
generate_TDR_offer                  8
generate_nonTDR_extension_offer     8


---
## Export to Excel

In [30]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from datetime import date

# ── palette (charcoal + semantic status colours only) ─────────────────────
C_HEADER  = "1F2937"  # dark charcoal
C_SUBHDR  = "374151"  # mid charcoal
C_PASS    = "D1FAE5"  # light green tint
C_FAIL    = "FEE2E2"  # light red tint
C_ERROR   = "FEF3C7"  # light amber tint
C_ALT     = "F9FAFB"  # off-white alternate row
C_WHITE   = "FFFFFF"
C_BORDER  = "D1D5DB"  # light gray
C_MID     = "4B5563"  # mid gray text

thin = Side(style="thin", color=C_BORDER)
bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)

def hfont(sz=10): return Font(name="Calibri", size=sz, bold=True, color=C_WHITE)
def bfont(sz=10): return Font(name="Calibri", size=sz, color="111827")
def mfont(sz=9):  return Font(name="Consolas", size=sz, color=C_MID)
def bg(c):        return PatternFill(fill_type="solid", fgColor=c)
ctr  = Alignment(horizontal="center", vertical="center")
lft  = Alignment(horizontal="left",   vertical="center", wrap_text=True)

STATUS_BG   = {"PASS": bg(C_PASS), "FAIL": bg(C_FAIL), "ERROR": bg(C_ERROR)}
STATUS_FONT = {"PASS": Font(name="Calibri", size=10, bold=True, color="166534"),
               "FAIL": Font(name="Calibri", size=10, bold=True, color="9B1C1C"),
               "ERROR": Font(name="Calibri", size=10, bold=True, color="92400E")}

wb = Workbook()

# ══════════════════════════════════════════════════════════
# Sheet 1 — Summary
# ══════════════════════════════════════════════════════════
ws = wb.active
ws.title = "Summary"
ws.sheet_view.showGridLines = False
for col, w in [("A",2),("B",32),("C",12),("D",12),("E",12),("F",12),("G",12)]:
    ws.column_dimensions[col].width = w

# Title
ws.merge_cells("B2:G2")
ws["B2"] = "Unit Test Results — DebtMind Advisor Services"
ws["B2"].font = Font(name="Calibri", size=16, bold=True, color="111827")
ws["B2"].alignment = lft
ws.merge_cells("B3:G3")
ws["B3"] = (f"Generated: {date.today().strftime('%d %B %Y')}   "
             "·   modules: services/functions  ·  services/ai_agent")
ws["B3"].font = Font(name="Calibri", size=9, color=C_MID)
ws["B3"].alignment = lft

# KPI tiles
ws.row_dimensions[5].height = 44
for (c1,c2), label, val, color in [
    (("B","C"), "Total Tests",      total,          C_HEADER),
    (("D","E"), "Passed",           passed,         "166534"),
    (("F","G"), "Failed / Error",   failed+errors,  "9B1C1C"),
]:
    ws.merge_cells(f"{c1}5:{c2}5")
    c = ws[f"{c1}5"]
    c.value     = f"{label}\n{val}"
    c.font      = Font(name="Calibri", size=14, bold=True, color=C_WHITE)
    c.fill      = bg(color)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

ws.merge_cells("B6:G6")
ws["B6"] = f"Pass Rate: {100*passed/total:.1f}%"
ws["B6"].font = Font(name="Calibri", size=11, bold=True, color="166534")
ws["B6"].alignment = lft

# Group table
ws.row_dimensions[8].height = 20
for col, val in zip(["B","C","D","E","F","G"],
                    ["Module / Group","Total","PASS","FAIL","ERROR","Pass Rate"]):
    c = ws[f"{col}8"]
    c.value = val; c.font = hfont(); c.fill = bg(C_SUBHDR); c.alignment = ctr; c.border = bdr

gstat = df_results.groupby("Group")["Status"].value_counts().unstack(fill_value=0)
for col in ["PASS","FAIL","ERROR"]:
    if col not in gstat.columns: gstat[col] = 0
gstat["Total"]     = gstat.sum(axis=1)
gstat["Pass Rate"] = (gstat["PASS"] / gstat["Total"] * 100).round(1)

for ri, (grp, row) in enumerate(gstat.iterrows(), start=9):
    ws.row_dimensions[ri].height = 18
    rf = bg(C_ALT) if ri % 2 == 0 else bg(C_WHITE)
    for ci, (v, al) in enumerate(zip(
        [grp, int(row["Total"]), int(row["PASS"]), int(row["FAIL"]),
         int(row["ERROR"]), f"{row['Pass Rate']:.1f}%"],
        [lft, ctr, ctr, ctr, ctr, ctr]), start=2):
        c = ws.cell(row=ri, column=ci)
        c.value = v; c.fill = rf; c.alignment = al; c.border = bdr
        c.font = (Font(name="Calibri", size=10, color="166534", bold=True)
                  if (ci==4 and int(row["PASS"])==int(row["Total"]))
                  else bfont())

# ══════════════════════════════════════════════════════════
# Sheet 2 — Detailed Results
# ══════════════════════════════════════════════════════════
wd = wb.create_sheet("Detailed Results")
wd.sheet_view.showGridLines = False
for col, w in [("A",2),("B",26),("C",38),("D",20),("E",18),("F",18),("G",9),("H",32)]:
    wd.column_dimensions[col].width = w

HDR_COLS = ["B","C","D","E","F","G","H"]
HDR_VALS = ["Group","Test Name","Input","Expected","Actual","Status","Notes"]
wd.row_dimensions[1].height = 22
for col, val in zip(HDR_COLS, HDR_VALS):
    c = wd[f"{col}1"]
    c.value = val; c.font = hfont(); c.fill = bg(C_HEADER); c.alignment = ctr; c.border = bdr

for ri, row in enumerate(df_results.itertuples(index=False), start=2):
    wd.row_dimensions[ri].height = 16
    status  = row.Status
    is_alt  = (ri % 2 == 0)
    row_bg  = bg(C_ALT) if is_alt else bg(C_WHITE)
    vals    = [row.Group, row._1, row.Input, row.Expected, row.Actual, row.Status, row.Notes]
    for col_letter, v in zip(HDR_COLS, vals):
        c = wd.cell(row=ri, column=ord(col_letter) - ord("A") + 1)
        c.value = v; c.border = bdr
        if col_letter == "G":   # Status
            c.font = STATUS_FONT.get(status, bfont()); c.fill = STATUS_BG.get(status, bg(C_WHITE)); c.alignment = ctr
        elif col_letter in ("D","E","F"):  # Input/Expected/Actual
            c.font = mfont(); c.fill = row_bg; c.alignment = lft
        else:
            c.font = bfont(); c.fill = row_bg; c.alignment = lft

wd.freeze_panes = "B2"

out_path = os.path.join(os.getcwd(), "test_results.xlsx")
wb.save(out_path)
print(f"Saved → {out_path}")

Saved → /Users/tanut/Desktop/work/Advisor-api--DebtMind-Hackathon/tests/test_results.xlsx
